# 02. Data profiling and cleaning

**Input:** `data/raw/online_retail_II.xlsx` (never changed)
**Output:** `data/processed/sales_clean.csv`, `data/processed/returns_clean.csv`, `reports/cleaning_log.csv`, `reports/profiling_summary.csv`

Works in Google Colab and on a local computer (run it from the `notebooks/` folder).

In [ ]:
from pathlib import Path
import pandas as pd

try:                                        # Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/retail-project")
    RAW = Path("/content/drive/MyDrive/online_retail_II.xlsx")
except ImportError:                         # local computer: this notebook is in notebooks/
    BASE = Path("..")
    RAW = BASE / "data/raw/online_retail_II.xlsx"

PROCESSED = BASE / "data/processed"
REPORTS = BASE / "reports"
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
print("Raw file found:", RAW.exists())

## 1. Load the raw data

In [ ]:
sheets = pd.read_excel(RAW, sheet_name=None)
for name, d in sheets.items():
    print(name, d.shape, d["InvoiceDate"].min(), "to", d["InvoiceDate"].max())
df = pd.concat(sheets.values(), ignore_index=True)
print("Total:", df.shape)

## 2. Data profiling (before any cleaning)

What to look for in the output below:
- **StockCode** holds both numbers and text, so it must be read as text. If not, numeric codes are lost later.
- **Description** and **Customer ID** have missing values.
- Exact **duplicate** rows, and **cancelled** invoices (start with `C`).
- **Quantity** and **Price** with zero or negative values.

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
})
print(profile, "\n")
print("Exact duplicate rows:", df.duplicated().sum())
print("Cancelled invoices (start with C):", df["Invoice"].astype(str).str.startswith("C").sum(), "rows")
print("StockCode value types:")
print(df["StockCode"].map(lambda x: type(x).__name__).value_counts(), "\n")
print(df[["Quantity", "Price"]].describe().round(2), "\n")
print("Rows with Quantity <= 0:", (df["Quantity"] <= 0).sum(), "| Price <= 0:", (df["Price"] <= 0).sum())
print("Countries:", df["Country"].nunique())
profile.to_csv(REPORTS / "profiling_summary.csv")

## 3. Cleaning (every step is logged with row counts)

In [ ]:
df = df.rename(columns={
    "Invoice": "invoice", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "Price": "unit_price",
    "Customer ID": "customer_id", "Country": "country",
})
df["invoice"] = df["invoice"].astype(str)
df["stock_code"] = df["stock_code"].astype(str).str.strip()     # codes are numbers or text: keep as text

log = []
def log_step(step, reason, before, after):
    log.append({"step": step, "reason": reason, "rows_before": before, "rows_after": after})

log_step("Load raw data (both sheets combined)", "Two sheets cover Dec 2009 - Dec 2011", len(df), len(df))
print(df.shape)

**Step 1: exact duplicates.** First check where they come from, then remove them.

In [ ]:
dups = df[df.duplicated()]
in_overlap = dups["invoice_date"].between("2010-12-01", "2010-12-09 23:59:59")
print("Duplicates:", len(dups), "| inside 1-9 Dec 2010: ", in_overlap.sum(), "| elsewhere:", (~in_overlap).sum())

n = len(df)
df = df.drop_duplicates()
log_step("Remove exact duplicates",
         f"Identical rows would count a line twice. {in_overlap.sum()} are in 1-9 Dec 2010 (covered by both sheets), "
         f"{(~in_overlap).sum()} are repeated lines elsewhere",
         n, len(df))
print(n - len(df), "rows removed")

**Step 2: codes that are not products** (postage, fees, discounts, manual entries, gift vouchers).

In [ ]:
non_product = ["POST", "D", "M", "DOT", "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2",
               "S", "B", "CRUK", "C2", "TEST001", "TEST002", "PADS"]
code = df["stock_code"].str.upper()
is_non_product = code.isin(non_product) | code.str.startswith("GIFT_")

n = len(df)
df = df[~is_non_product]
log_step("Remove non-product codes",
         "Postage, fees, discounts, manual entries and gift vouchers are not product sales", n, len(df))
print(n - len(df), "rows removed")

**Step 3: price of zero or less.**

In [ ]:
n = len(df)
df = df[df["unit_price"] > 0]
log_step("Remove price <= 0", "Free items and accounting adjustments, not real sales", n, len(df))
print(n - len(df), "rows removed")

**Step 4: cancelled orders** (invoice starts with C) go to a separate returns table, so we can calculate net revenue.

In [ ]:
is_cancel = df["invoice"].str.startswith("C")
returns = df[is_cancel].copy()
n = len(df)
df = df[~is_cancel].copy()
log_step("Split cancelled orders into returns table",
         "Invoice starts with C. Kept separately so we can calculate net revenue", n, len(df))

n = len(df)
df = df[df["quantity"] > 0].copy()
log_step("Remove quantity <= 0 in sales",
         f"Checked: {n - len(df)} rows. Stock write-offs and corrections would be removed here", n, len(df))
print("sales:", len(df), "| returns:", len(returns))

**Step 5: product names, customer ID and revenue.**

In [ ]:
df["description"] = df["description"].str.strip()
returns["description"] = returns["description"].str.strip()

main_desc = (df.dropna(subset=["description"])
               .groupby("stock_code")["description"]
               .agg(lambda s: s.mode().iloc[0]))
df["description"] = df["stock_code"].map(main_desc)
returns["description"] = returns["stock_code"].map(main_desc)

for d in (df, returns):
    d["customer_id"] = d["customer_id"].astype("Int64")
    d["revenue"] = d["quantity"] * d["unit_price"]

log_step("Standardize descriptions", "One name per product code (most frequent name)", len(df), len(df))
log_step("Keep rows with missing Customer ID",
         f"{df['customer_id'].isna().sum()} rows. Valid for sales by product/country/time; excluded later from customer analysis (RFM, repeat rate)",
         len(df), len(df))

**Step 6: outliers.** Look at the biggest orders and returns before deciding.

In [ ]:
cols = ["invoice", "description", "quantity", "unit_price", "revenue"]
print(df.nlargest(5, "revenue")[cols], "\n")
print(returns.nsmallest(5, "revenue")[cols], "\n")

picnic = df[df["description"] == "PICNIC BASKET WICKER SMALL"]
print(picnic[picnic["unit_price"] > 500][["invoice", "quantity", "unit_price", "revenue", "customer_id"]])
n_picnic_returns = returns["stock_code"].isin(picnic["stock_code"]).sum()
print("Returns for this product:", n_picnic_returns)

log_step("Outliers: reviewed, not removed",
         "The biggest orders (80,995 and 74,215 units) were cancelled in full and cancel out through the returns table. "
         f"The 60-unit picnic basket order (price 649.50) has {n_picnic_returns} returns, so it is kept as a real large order",
         len(df), len(df))

## 4. Checks before saving

Expected for the Online Retail II data: sales 1,003,340 rows, returns 17,914 rows, gross sales 19,643,862, returns -716,426, net revenue 18,927,436.

In [ ]:
assert df["stock_code"].notna().all()
assert df["description"].notna().all()
assert (df["quantity"] > 0).all() and (df["unit_price"] > 0).all()

print(df.shape, returns.shape)
print(df.isna().sum())
print(df["invoice_date"].min(), df["invoice_date"].max())

gross = df["revenue"].sum()
ret = returns["revenue"].sum()
print(f"Gross sales: {gross:,.0f} | Returns: {ret:,.0f} | Net revenue: {gross + ret:,.0f}")

## 5. Save the clean data and the cleaning log

In [ ]:
out_cols = ["invoice", "stock_code", "description", "quantity", "invoice_date",
            "unit_price", "customer_id", "country", "revenue"]
df[out_cols].to_csv(PROCESSED / "sales_clean.csv", index=False)
returns[out_cols].to_csv(PROCESSED / "returns_clean.csv", index=False)
pd.DataFrame(log).to_csv(REPORTS / "cleaning_log.csv", index=False)
pd.DataFrame(log)